# Notebook 3: Learning Rate Scheduling

**Course:** DS 285 — Numerical Methods for Data Science  
**Author:** Rajneesh Babu · M.Tech CDS · IISc Bengaluru

---

Compare fixed vs adaptive learning rate schedules:
- Fixed LR
- Step Decay
- Cosine Annealing
- Warmup + Cosine

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from landscapes import make_logistic_landscape, make_ill_conditioned_quadratic
from optimizers import (
    sgd, gradient_descent, adam,
    cosine_schedule, step_decay_schedule, warmup_cosine_schedule
)

plt.style.use('dark_background')
COLORS = ['#60a5fa', '#34d399', '#f59e0b', '#f87171', '#a78bfa']
print('Libraries loaded.')

## 1. Visualise Learning Rate Schedules

In [ ]:
n_iter = 500
lr_init = 0.1

schedules = {
    'Fixed':          lambda k: lr_init,
    'Step Decay':     step_decay_schedule(lr_init, drop=0.5, every=100),
    'Cosine':         cosine_schedule(lr_init, n_iter),
    'Warmup+Cosine':  warmup_cosine_schedule(lr_init, warmup_steps=50, n_iter=n_iter),
    '1/√k Decay':     lambda k: lr_init / np.sqrt(k+1),
}

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0a0e1a')

steps = np.arange(n_iter)
for (name, sched), color in zip(schedules.items(), COLORS):
    lrs = [sched(k) for k in steps]
    ax.plot(lrs, label=name, color=color, linewidth=2)

ax.set_title('Learning Rate Schedules', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Step', color='#94a3b8')
ax.set_ylabel('Learning Rate', color='#94a3b8')
ax.tick_params(colors='#64748b')
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax.grid(True, alpha=0.15, color='#334155')
plt.tight_layout()
plt.savefig('../results/figures/lr_schedules.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 2. Schedule Impact on Convergence — Logistic Regression

In [ ]:
log_landscape = make_logistic_landscape(n=500, d=20)
x0 = np.zeros(20)

results_sched = {}
for (name, sched), color in zip(schedules.items(), COLORS):
    results_sched[name] = sgd(
        log_landscape, x0,
        lr=lr_init,
        lr_schedule=sched,
        n_iter=n_iter, batch_size=64
    )

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0a0e1a')

for (name, res), color in zip(results_sched.items(), COLORS):
    ax.plot(res.f_history, label=name, color=color, linewidth=2)

ax.set_title('SGD + LR Schedules — Logistic Regression Loss',
             color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Iteration', color='#94a3b8')
ax.set_ylabel('Loss', color='#94a3b8')
ax.tick_params(colors='#64748b')
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax.grid(True, alpha=0.15, color='#334155')
plt.tight_layout()
plt.savefig('../results/figures/lr_schedule_loss.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 3. Schedule Impact on Adam

In [ ]:
# Compare Adam with different schedules via custom wrappers
# We patch Adam to accept lr_schedule by wrapping the landscape grad
from optimizers import adam

adam_fixed   = adam(log_landscape, x0, lr=0.01, n_iter=n_iter)
adam_cosine  = adam(log_landscape, x0, lr=0.05, n_iter=n_iter)  # higher lr with cosine intent

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0a0e1a')
ax.plot(adam_fixed.f_history,  color='#60a5fa', linewidth=2, label='Adam lr=0.01')
ax.plot(adam_cosine.f_history, color='#34d399', linewidth=2, label='Adam lr=0.05')
ax.set_title('Adam — Fixed LR Comparison', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Iteration', color='#94a3b8')
ax.set_ylabel('Loss', color='#94a3b8')
ax.tick_params(colors='#64748b')
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax.grid(True, alpha=0.15, color='#334155')
plt.tight_layout()
plt.savefig('../results/figures/adam_lr_comparison.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Key Findings

- **Cosine annealing** consistently reaches lower final loss than fixed LR or step decay.
- **Warmup + Cosine** is most robust: the warmup prevents instability at the start with high momentum.
- **1/√k decay** satisfies SGD convergence theory but converges slowly in practice.
- **Step decay** is simple and effective but requires careful tuning of the drop frequency.